[< Back to Main README](../README.md) | [Demo README](./README.md)

# Neurosymbolic Integration: Enforcing Business Rules with Strands Hooks

Based on: [ATA: Autonomous Trustworthy Agents](https://arxiv.org/html/2510.16381v1)

## The Problem

AI agents hallucinate when business rules are expressed only in natural language prompts:

- **Parameter errors**: Agent calls `book_hotel(guests=15)` despite "Maximum 10 guests" in docstring
- **Completeness errors**: Agent executes bookings without required payment verification
- **Tool bypass behavior**: Agent confirms success without calling validation tools

**Why prompt engineering fails**: Prompts are suggestions, not constraints. Agents can ignore docstring instructions because they're processed as text, not executable rules.

## The Solution: Strands Hooks for Neurosymbolic Validation

Strands Agents provides **hooks**—a composable system that intercepts tool calls before execution to enforce symbolic rules:

```
User Query → LLM (understands) → Tool Selection → Hook (validates) → Execute or Block
```

**Key insight**: Use Strands hooks to validate tool calls before execution. The LLM cannot bypass rules enforced at the framework level.

## What This Demo Shows

We run **the same 3 scenarios** on two agents side by side:

| Agent | Guardrails |
|-------|-----------|
| **Baseline** | No hook — standard agent |
| **Neurosymbolic** | Hook intercepts every tool call |

Scenarios tested:
1. Confirm a booking without payment → should be **blocked**
2. Book a hotel for 15 guests (max 10) → should be **blocked**
3. Valid booking for 5 guests → should be **allowed**

The hook blocks invalid operations before tools execute, preventing hallucinations.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

## Configure AWS Credentials

This demo uses Amazon Bedrock (default model provider for Strands Agents). Ensure your AWS credentials are configured.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [ ]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os
if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

import os
# Verify AWS credentials are available
# boto3 is the AWS SDK for Python. We use it to verify your AWS credentials.
# sts (Security Token Service) provides the GetCallerIdentity API
# which confirms your credentials are valid without requiring any specific permissions.
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# To use OpenAI instead of Bedrock:
#   pip install "strands-agents[openai]"
#   os.environ["OPENAI_API_KEY"] = "your-key-here"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")


## Setup

In [ ]:
from strands import Agent, tool
# HookProvider: base class for defining lifecycle hooks in Strands Agents
# HookRegistry: manages hook registration; hooks attach to specific lifecycle events
# BeforeToolCallEvent: the event fired just before a tool is executed — used to inspect and cancel tool calls
from strands.hooks import HookProvider, HookRegistry, BeforeToolCallEvent
# Model configuration — Amazon Bedrock (default, no extra import needed)
# Strands Agents uses Bedrock by default when no model is specified.
#
# To use a specific Bedrock model:
#   MODEL = "us.anthropic.claude-sonnet-4-20250514-v1:0"
#   agent = Agent(tools=..., model=MODEL)
#
# To use OpenAI instead:
#   pip install "strands-agents[openai]"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")
#
# See all providers: https://strandsagents.com/docs/user-guide/concepts/model-providers/
from datetime import datetime
from rules import BOOKING_RULES, CONFIRMATION_RULES, validate



# Simulated state
STATE = {
    "bookings": {"BK001": {"hotel": "Grand Hotel", "check_in": "2026-02-15", "guests": 2}},
    "payments": {}  # No payments verified yet
}

print("✅ Setup complete")

## Step 1: Define Symbolic Rules

Business rules as verifiable code. These rules cannot be bypassed by prompt engineering.

In [ ]:
from rules import BOOKING_RULES, CONFIRMATION_RULES

print("Booking Rules:")
for rule in BOOKING_RULES:
    print(f"  - {rule.name}: {rule.message}")

print("\nConfirmation Rules:")
for rule in CONFIRMATION_RULES:
    print(f"  - {rule.name}: {rule.message}")

## Step 2: Create Validation Hook

The hook intercepts tool calls before execution using `BeforeToolCallEvent`. If rules are violated, it cancels the tool with `event.cancel_tool`.

In [ ]:
class NeurosymbolicHook(HookProvider):
    """Validates tool calls against symbolic rules before execution"""
    
    def __init__(self, state: dict):
        self.state = state
        self.rules = {
            "book_hotel": BOOKING_RULES,
            "confirm_booking": CONFIRMATION_RULES,
        }
        self.blocked_calls = []  # Track every blocked tool call
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.validate)
    
    def validate(self, event: BeforeToolCallEvent) -> None:
        tool_name = event.tool_use["name"]
        if tool_name not in self.rules:
            return
        
        ctx = self._build_context(tool_name, event.tool_use["input"])
        passed, violations = validate(self.rules[tool_name], ctx)
        
        if not passed:
            reason = f"BLOCKED: {', '.join(violations)}"
            event.cancel_tool = reason
            self.blocked_calls.append({
                "tool": tool_name,
                "params": event.tool_use["input"],
                "reason": reason,
            })
    
    def reset(self):
        """Clear blocked_calls log between test runs"""
        self.blocked_calls = []
    
    def _build_context(self, tool_name: str, params: dict) -> dict:
        if tool_name == "book_hotel":
            try:
                ci = datetime.strptime(params["check_in"], "%Y-%m-%d")
                co = datetime.strptime(params["check_out"], "%Y-%m-%d")
                return {
                    "check_in": ci,
                    "check_out": co,
                    "guests": params.get("guests", 1),
                    "days_until_checkin": (ci - datetime.now()).days
                }
            except (ValueError, KeyError):
                return {
                    "check_in": None,
                    "check_out": None,
                    "guests": 999,
                    "days_until_checkin": -999
                }
        elif tool_name == "confirm_booking":
            return {"payment_verified": params["booking_id"] in self.state["payments"]}
        return {}

print("✅ Hook created (with blocked_calls tracking)")

## Step 3: Define Clean Tools

Tools have no validation logic—they're simple and focused on business operations. The hook handles all validation.

In [ ]:
@tool
def book_hotel(hotel: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Book a hotel room."""
    return f"SUCCESS: Booked {hotel} for {guests} guests, {check_in} to {check_out}"

@tool
def process_payment(amount: float, booking_id: str) -> str:
    """Process payment for a booking."""
    if booking_id not in STATE["bookings"]:
        return "ERROR: Booking not found"
    STATE["payments"][booking_id] = amount
    return f"SUCCESS: Processed ${amount} for {booking_id}"

@tool
def confirm_booking(booking_id: str) -> str:
    """Confirm a booking."""
    return f"SUCCESS: Confirmed {booking_id}"

print("✅ Tools defined")

## Step 4: Create Agents for Comparison

We create two agents with identical tools and prompts — the only difference is the hook:

| Agent | Hook | Guardrails |
|-------|------|-----------|
| `baseline_agent` | None | ❌ No validation |
| `guarded_agent` | `NeurosymbolicHook` | ✅ Rules enforced at framework level |

In [ ]:
SCENARIOS = [
    {
        "name": "Confirm booking without payment",
        "query": "Confirm booking BK001",
        "expected": "BLOCKED",
        "rule": "Payment must be verified before confirmation",
    },
    {
        "name": "Book hotel exceeding guest limit (15 > max 10)",
        "query": "Book Grand Hotel for 15 people from 2026-03-20 to 2026-03-25",
        "expected": "BLOCKED",
        "rule": "Maximum 10 guests per booking",
    },
    {
        "name": "Valid booking (5 guests, future dates)",
        "query": "Book Grand Hotel for 5 guests from 2026-03-20 to 2026-03-25",
        "expected": "ALLOWED",
        "rule": "All rules pass",
    },
]

# Agent WITHOUT guardrails (baseline)
baseline_agent = Agent(
    tools=[book_hotel, process_payment, confirm_booking],
    
)

# Agent WITH neurosymbolic guardrails
hook = NeurosymbolicHook(STATE)
guarded_agent = Agent(
    tools=[book_hotel, process_payment, confirm_booking],
    hooks=[hook],
    
)

W = 60          # output width for screenshots
results = []    # collect per-test results for comparison cell

print("✅ Two agents created:")
print("   baseline_agent — no hook, no validation")
print("   guarded_agent  — NeurosymbolicHook attached")
print(f"\n📋 {len(SCENARIOS)} test scenarios:")
for i, s in enumerate(SCENARIOS, 1):
    print(f"   {i}. [{s['expected']:7}] {s['name']}")

## Test 1: Confirm Booking Without Payment

**Expected**: Hook blocks — payment is not verified.

In [ ]:
def run_test(scenario, index):
    """Run one scenario on both agents and print a side-by-side comparison."""
    s = scenario
    STATE["payments"] = {}
    hook.reset()

    q = s["query"]
    q_short = (q[:50] + "...") if len(q) > 50 else q

    print("═" * W)
    print(f"TEST {index}: {s['name'].upper()}")
    print("═" * W)
    print(f"\n  Query:  \"{q_short}\"")
    print(f"  Rule:   {s['rule']}")

    # ── BASELINE ──────────────────────────────────────
    print(f"\n  ── BASELINE  (no hook)\n")
    result_b = baseline_agent(s["query"])
    result_b_str = str(result_b)
    executed = any(
        kw in result_b_str.upper()
        for kw in ["SUCCESS", "BOOKED", "CONFIRMED", "PROCESSED"]
    )
    resp_b = (result_b_str[:80] + "...") if len(result_b_str) > 80 else result_b_str
    if executed:
        print(f"  🔴  EXECUTED — no guardrail")
    else:
        print(f"  ⚠️   No tool called")
    print(f"  \"{resp_b}\"")

    # ── GUARDED ───────────────────────────────────────
    print(f"\n  {'─' * (W - 2)}\n")
    print(f"  ── GUARDED   (neurosymbolic hook)\n")
    hook.reset()
    result_g = guarded_agent(s["query"])
    result_g_str = str(result_g)
    blocked = len(hook.blocked_calls) > 0
    resp_g = (result_g_str[:80] + "...") if len(result_g_str) > 80 else result_g_str

    if blocked:
        call = hook.blocked_calls[0]
        params_str = str(call["params"])
        if len(params_str) > 45:
            params_str = params_str[:45] + "..."
        print(f"  ✅  BLOCKED by hook")
        print(f"  Tool:   {call['tool']}({params_str})")
        print(f"  Reason: {call['reason']}")
    else:
        print(f"  🟢  ALLOWED — all rules passed")
    print(f"  \"{resp_g}\"")

    print(f"\n{'═' * W}")

    # Store for comparison table
    results.append({
        "name": s["name"],
        "expected": s["expected"],
        "executed": executed,
        "blocked": blocked,
    })


run_test(SCENARIOS[0], 1)

## Test 2: Book Hotel Exceeding Guest Limit

**Expected**: Hook blocks — 15 guests exceeds the maximum of 10.

In [ ]:
run_test(SCENARIOS[1], 2)

## Test 3: Valid Booking

**Expected**: Both agents execute — all rules pass.

In [ ]:
run_test(SCENARIOS[2], 3)

In [ ]:
SHORT_NAMES = [
    "Confirm without payment",
    "Book 15 guests (max 10)",
    "Valid booking (5 guests)",
]

print("═" * W)
print("COMPARISON RESULTS")
print("═" * W)

print(f"\n  {'Scenario':<27} {'Expected':>8}  {'Baseline':>10}  {'Guarded':>10}")
print(f"  {'─' * 56}")

baseline_correct = 0
guarded_correct  = 0

for i, r in enumerate(results):
    expected_blocked = r["expected"] == "BLOCKED"

    base_ok  = (expected_blocked and not r["executed"]) or \
               (not expected_blocked and r["executed"])
    guard_ok = (expected_blocked and r["blocked"]) or \
               (not expected_blocked and not r["blocked"])

    baseline_correct += base_ok
    guarded_correct  += guard_ok

    b_label = "✅ correct" if base_ok  else "❌ wrong  "
    g_label = "✅ correct" if guard_ok else "❌ wrong  "

    print(f"  {SHORT_NAMES[i]:<27} {r['expected']:>8}  {b_label}  {g_label}")

print(f"  {'─' * 56}")
print(f"  {'TOTAL':<27} {'':>8}  {baseline_correct}/{len(results)} correct  "
      f"{guarded_correct}/{len(results)} correct")

invalid_total   = sum(1 for r in results if r["expected"] == "BLOCKED")
blocked_correct = sum(1 for r in results if r["expected"] == "BLOCKED" and r["blocked"])
valid_total     = sum(1 for r in results if r["expected"] == "ALLOWED")
allowed_correct = sum(1 for r in results if r["expected"] == "ALLOWED" and not r["blocked"])

print(f"\n  Hook blocked {blocked_correct}/{invalid_total} invalid operations")
print(f"  Hook allowed {allowed_correct}/{valid_total} valid operations")
print(f"\n{'═' * W}")

## Comparison: All Tests

## Key Findings

| Scenario | Without Guardrails | With Neurosymbolic Hook |
|----------|-------------------|------------------------|
| Confirm booking (no payment) | ❌ Tool executes → hallucination | ✅ Blocked before execution |
| Book 15 guests (max 10) | ❌ Tool executes → rule violated | ✅ Blocked before execution |
| Valid booking (5 guests) | ✅ Tool executes | ✅ Tool executes |

### Why the Hook Wins

Without guardrails, business rules live in docstrings. The LLM reads them but is never forced to respect them — it can confirm a booking, exceed guest limits, or skip payment checks entirely.

With a `NeurosymbolicHook`:
- Rules are **code**, not text
- Validation runs **before** the tool executes
- The LLM receives a cancellation message it cannot override
- **Zero hallucinations** on rule violations — guaranteed at the framework level

### The Pattern

```
Without hook:  User → LLM decides → Tool executes (no guardrail)
With hook:     User → LLM decides → Hook validates → Execute OR Block
                                           ↑
                                    symbolic rules
                                    (cannot be bypassed)
```

### References

- [ATA: Autonomous Trustworthy Agents](https://arxiv.org/html/2510.16381v1)
- [Strands Agents Hooks documentation](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/)